In [ ]:
!pip install -q -U transformers datasets

In [ ]:
# load raw data
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/riccardotella/Sentiment_Analysis_of_Financial_News/main/data/processed/preprocessed_texts.csv")

X = df["text"]
y = df["label"]

# encode labels to ints
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_enc = le.fit_transform(y)      # neg/neu/pos -> 0/1/2
print(le.classes_)

# split 80/20
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)

print(len(X_train), len(X_test))
print(le.classes_)

In [ ]:
# tokenizer (DistilBERT)
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# check one example
example = X_train.iloc[0]
print("ORIGINAL:", example)
print("TOKENS:", tokenizer.tokenize(example))
print("IDS:", tokenizer(example)["input_ids"])

In [ ]:
# build datasets and tokenize
from datasets import Dataset

train_ds = Dataset.from_dict({"text": list(X_train), "labels": list(y_train)})
test_ds  = Dataset.from_dict({"text": list(X_test),  "labels": list(y_test)})

def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

In [ ]:
# load pretrained model
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

In [ ]:
# metrics
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

# training args + trainer
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    logging_steps=50,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

In [ ]:
# train
trainer.train()

In [ ]:
# evaluate
results = trainer.evaluate()
print(results)

In [ ]:
# per-class report + confusion matrix
import numpy as np
preds_output = trainer.predict(test_ds)
y_pred_t5 = np.argmax(preds_output.predictions, axis=1)

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_pred_t5, target_names=le.classes_))
print(confusion_matrix(y_test, y_pred_t5))

In [ ]:
# binary: drop neutral
from sklearn.preprocessing import LabelEncoder

df_bin = df[df["label"] != "neutral"]

le_bin = LabelEncoder()
y_bin_enc = le_bin.fit_transform(df_bin["label"])
print(le_bin.classes_)

X_bin = df_bin["text"]
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_bin, y_bin_enc, test_size=0.2, random_state=42, stratify=y_bin_enc)

In [ ]:
# tokenize binary
train_ds_b = Dataset.from_dict({"text": list(X_train_b), "labels": list(y_train_b)})
test_ds_b  = Dataset.from_dict({"text": list(X_test_b),  "labels": list(y_test_b)})
train_ds_b = train_ds_b.map(tokenize_fn, batched=True)
test_ds_b  = test_ds_b.map(tokenize_fn, batched=True)

In [ ]:
# new model, 2 labels
model_bin = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

In [ ]:
# trainer + train (binary)
trainer_bin = Trainer(
    model=model_bin,
    args=training_args,
    train_dataset=train_ds_b,
    eval_dataset=test_ds_b,
    compute_metrics=compute_metrics,
)
trainer_bin.train()

In [ ]:
# evaluate binary
preds_b = trainer_bin.predict(test_ds_b)
y_pred_b = np.argmax(preds_b.predictions, axis=1)

print(classification_report(y_test_b, y_pred_b, target_names=le_bin.classes_))
print(confusion_matrix(y_test_b, y_pred_b))